In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/amanatar/ans-verifys/__results__.html
/kaggle/input/notebooks/amanatar/ans-verifys/vllm_server.log
/kaggle/input/notebooks/amanatar/ans-verifys/__notebook__.ipynb
/kaggle/input/notebooks/amanatar/ans-verifys/__output__.json
/kaggle/input/notebooks/amanatar/ans-verifys/custom.css
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/model.safetensors.index.json
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/model-00003-of-00004.safetensors
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/config.json
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/LICENSE
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/model-00001-of-00004.safetensors
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/README.md
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/USE_POLICY.md
/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1/tokenizer.json
/kaggl

In [2]:
#module2
import json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


data = []

with open("/kaggle/input/datasets/ritesh6107/finaldata/final_cleaned.jsonl") as f:
    for line in f:
        data.append(json.loads(line))

print("Total data:", len(data))




texts = []

for item in data:
    text = item["instruction"] + "\n" + item["output"]
    texts.append(text)


print(texts[100])
len(texts)

Total data: 175723
patient: hi,i am 34 yrs old female. since the past 10 days i am suffering from chest congestion . i feel a slight pain in breathing and even when i bend down to pick up things. i even feel tired and out of breath after a short talk or walk. i consulted a doc and did my ecg. the report says every thing is normal. since the past three days i caught cold , and the recent development is that apart from the above mentioned symptons i feel a pain on the right side of my back when breathing.. suggest me what to do and why is this happening..
disease: general medical condition solution: hello dear , hiwarm welcome to healthcaremagic.comi have gone through your query thoroughly .* the possible reasons are - respiratory infection - musuclo skeletal ache - intercostal nerve bundle related - low hemoglobin - rib pathology or inflammation others .* needs assessment with x-ray chest , lab tests for further management .hope this will clear your query .


175723

In [ ]:
# def chunk_text(text, chunk_size=500, overlap=50):
#     words = text.split()
#     chunks = []
    
#     for i in range(0, len(words), chunk_size - overlap):
#         chunk = " ".join(words[i:i+chunk_size])
#         chunks.append(chunk)
    
#     return chunks

# chunked_texts = []

# for t in texts:
#     chunks = chunk_text(t)
#     chunked_texts.extend(chunks)

# print("Total chunks:", len(chunked_texts))




In [3]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.8 MB/s eta 0:00:00:00:0100:01


In [4]:
embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
embeddings = embed_model.encode(
    texts,
    batch_size=256,   # 🔥 change from 256 → 16
    normalize_embeddings=True,
    show_progress_bar=True
)

In [ ]:
import faiss
import numpy as np



embeddings = np.array(embeddings).astype("float32")

# dimension
d = embeddings.shape[1]

# number of clusters
nlist = 1000  

# quantizer (base index)
quantizer = faiss.IndexFlatIP(d)

# main FAISS index (IMPORTANT VARIABLE)
index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# train (mandatory)
index.train(embeddings)

# add vectors
index.add(embeddings)

# search tuning
index.nprobe = 50

print("FAISS index ready:", index)

In [6]:
print(index.ntotal)
print("texts size:", len(texts))

175723
texts size: 175723


In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [8]:
import faiss
import numpy as np
import json
import random

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from sklearn.metrics.pairwise import cosine_similarity

In [9]:
model_path = "/kaggle/input/models/ritesh6107/llama3b-instruct/pytorch/default/1"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("Models loaded")






   

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Models loaded


In [10]:
def clean_context(results):
    clean = []
    for r in results:
        txt = r.lower()

        if any(x in txt for x in ["ligament", "injury"]):
            continue

        clean.append(r)

    return clean

In [11]:
def retrieve(query, k=40, top_k=3):

    # query embedding
    q_vec = embed_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")

    
    D, I = index.search(q_vec, k)

  
    results = [texts[i] for i in I[0] if i != -1]

   
    results = clean_context(results)

    # reranker
    pairs = [[query, r] for r in results]
    scores = reranker.predict(pairs)

    
    boosted = []
    for score, text in zip(scores, results):
        extra = 0
        t = text.lower()
        if "behind the eyes" in t:
            extra += 0.5
        if "joint" in t and "fever" in t:
            extra += 0.3
        boosted.append(score + extra)

  
    ranked = sorted(zip(boosted, results), reverse=True)
    final_results = [r for _, r in ranked[:top_k]]

    return final_results

In [12]:
def generate_answer(query):

    context_list = retrieve(query, k=40, top_k=3)
    context = "\n\n".join(context_list)
    context = context[:1200]

    prompt = f"""
You are a medical assistant.

Rules:
1. Use the provided context, but DO NOT blindly trust it.
2. If the context is weak, generic, or unrelated, prioritize symptoms.
3. Always choose the MOST COMMON and MOST PROBABLE disease.
4. Avoid rare or unrelated diseases unless clearly supported.
5. Avoid generic labels like "viral fever" if a more specific disease fits.
6. Provide ONLY ONE final answer.
7. Keep the answer simple, clear, and concise.
8. Fill ALL fields properly. Do not leave anything empty.
9. Provide ONLY ONE final answer.

Context:
{context}

Question:
{query}

Output Format:
Disease:
Causes:
Symptoms:
Treatment:
Prevention:
Consultation:
"""

    result = pipe(
        prompt,
        max_new_tokens=120,
        temperature=0.2,
        do_sample=False
    )

    output = result[0]["generated_text"]

    return output

In [15]:
print(generate_answer("fever and headache"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=120) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a medical assistant.

Rules:
1. Use the provided context, but DO NOT blindly trust it.
2. If the context is weak, generic, or unrelated, prioritize symptoms.
3. Always choose the MOST COMMON and MOST PROBABLE disease.
4. Avoid rare or unrelated diseases unless clearly supported.
5. Avoid generic labels like "viral fever" if a more specific disease fits.
6. Provide ONLY ONE final answer.
7. Keep the answer simple, clear, and concise.
8. Fill ALL fields properly. Do not leave anything empty.
9. Provide ONLY ONE final answer.

Context:
patient: hello doctor,i am having headache and fever for four days and i had sumo cold and gelusil. now, the fever is less but the headache continues.
disease: infection or viral fever solution: hello. a headache along with fever is a common symptom of viral fever. since your fever has come down, your headache should go away or get better in a day or so. drink plenty of fluids and stay hydrated and take rest. you can take paracetamol 500 mg on a fu

In [14]:
print(texts[9])

patient: my calf on left leg is sore to touch and when i bend leg. it is slightly swollen, not red or warm to touch, no fever. it s only sore in one spot. i wore a pair of knee high socks the other day that were very tight and when i took them off my legs were sore but right leg is not sore now. ??
disease: infection or viral fever solution: i wish you meet a vascular surgeon. as he might be able to provide proper inputs to your situation. little dagnositic procedure and little of physical exercise should be of help.


In [15]:
def clean(text):
    return text.lower().strip()


def keyword_score(pred, exp):
    pred = pred.lower()
    exp = exp.lower()

    exp_words = set(exp.split())
    pred_words = set(pred.split())

    return len(exp_words & pred_words) / max(len(exp_words), 1)


def semantic_score(pred, exp):
    emb1 = embed_model.encode([pred])
    emb2 = embed_model.encode([exp])
    
    return cosine_similarity(emb1, emb2)[0][0]


def structure_bonus(pred):
    pred = pred.lower()
    
    sections = ["disease:", "causes:", "symptoms:", "treatment:"]
    
    count = sum([1 for s in sections if s in pred])
    
    return count / 4


def final_score(pred, exp):
    k = keyword_score(pred, exp)
    s = semantic_score(pred, exp)
    st = structure_bonus(pred)

    return 0.5*k + 0.3*s + 0.2*st


In [16]:
sample_size = 1000

sample_data = random.sample(data, sample_size)

questions = [item["instruction"] for item in sample_data]
expected = [item["output"] for item in sample_data]

print("Sample loaded:", len(questions))

Sample loaded: 1000


In [ ]:
batch_size = 25
scores = []

for i in range(0, len(questions), batch_size):

    batch_q = questions[i:i+batch_size]
    batch_exp = expected[i:i+batch_size]

    preds = [generate_answer(q) for q in batch_q]

    batch_scores = []

    for p, e in zip(preds, batch_exp):
        score = final_score(p, e)
        
        scores.append(score)
        batch_scores.append(score)

    print(f"""
Batch {i}-{i+batch_size}
Batch Accuracy: {np.mean(batch_scores):.2f}
Overall Accuracy: {np.mean(scores):.2f}
""")

print("\n Final Accuracy:", np.mean(scores))
